# Notebook 02 — Running ColabFold on PD-L1

**Learning objectives:**
- Run AlphaFold2 (via ColabFold) on the PD-L1 protein sequence
- Read and interpret pLDDT, PAE, and pTM output
- Compare the AF2 prediction to the experimental X-ray structure (5C3T)
- Understand what high-confidence prediction looks like

**Time:** ~45 minutes (5 min setup, 5 min running, 35 min analysis)

**Companion wiki page:** [4.3 Running ColabFold](https://rucha1796.github.io/internship-bindcraft-2026/m4_03_colabfold/)

---
> **GPU required.** Go to **Runtime → Change runtime type → T4 GPU** before running Cell 1.

## Cell 1 — Install ColabFold

This takes about 3–5 minutes. Wait for the ✓ before proceeding.

In [ ]:
%%bash
# Install ColabFold (MMseqs2-based, fast MSA generation)
pip install -q "colabfold[alphafold-minus-jax] @ git+https://github.com/sokrypton/ColabFold"
pip install -q --upgrade jax jaxlib
pip install -q py3Dmol

# Download AlphaFold2 parameters (if not already cached)
python -c "from colabfold.download import download_alphafold_params; download_alphafold_params('AlphaFold2-multimer-v3', '.')" 2>/dev/null || echo "Parameters already available"
echo "✓ ColabFold installation complete"

## Cell 2 — Define the PD-L1 sequence

This is the extracellular domain of human PD-L1 (residues 19–238, signal peptide removed). This is the part that interacts with PD-1.

In [ ]:
# Human PD-L1 extracellular domain (UniProt Q9NZQ7, residues 19-238)
PDL1_SEQUENCE = "MIFALLAALAVFPVSQAMQTEVDGTQNLTICRFPVEKQLDLAALIVYWEMEDKNIIQFVHGEEDLKVQHSSYRQRARLLKDQLSLGNAALQITDVKLQDAGVYRCMISYGGADYKRITVKVNAPYNKINQRILVDPQRLYLVHNQKLCILHQKFTTL"

print(f"Protein name: Human PD-L1 extracellular domain")
print(f"Sequence length: {len(PDL1_SEQUENCE)} residues")
print(f"Sequence:")
print(PDL1_SEQUENCE)
print()

# Count amino acid composition
from collections import Counter
aa_counts = Counter(PDL1_SEQUENCE)
print("Amino acid composition (top 10):")
for aa, count in sorted(aa_counts.items(), key=lambda x: -x[1])[:10]:
    print(f"  {aa}: {count} ({count/len(PDL1_SEQUENCE)*100:.1f}%)")

## Cell 3 — Run ColabFold prediction

This runs AlphaFold2 using MMseqs2 for fast MSA generation. Generates 5 models.

**Expected time:** ~3–5 minutes on T4 GPU

In [ ]:
import os

# Create output directory
os.makedirs("colabfold_output", exist_ok=True)

# Write sequence to FASTA file
with open("PDL1.fasta", "w") as f:
    f.write(">PDL1_extracellular\n")
    f.write(PDL1_SEQUENCE + "\n")

print("Running ColabFold prediction...")
print("This will take 3-5 minutes.\n")

# Run ColabFold
!colabfold_batch PDL1.fasta colabfold_output/ \
    --num-recycle 3 \
    --num-models 5 \
    --model-type alphafold2_ptm \
    2>&1 | grep -E '(model|pLDDT|pTM|Done|Error|warning)' || true

print("\n✓ ColabFold complete")

## Cell 4 — Check the output files

In [ ]:
import os

output_files = sorted(os.listdir("colabfold_output"))
print("Files generated by ColabFold:")
for f in output_files:
    size = os.path.getsize(f"colabfold_output/{f}")
    print(f"  {f:60s} ({size:>8,} bytes)")

## Cell 5 — Read pLDDT and pTM scores

In [ ]:
import json
import glob

# Find all JSON score files
score_files = sorted(glob.glob("colabfold_output/*scores*.json"))

print(f"Found {len(score_files)} model score files:\n")

all_scores = []
for sf in score_files:
    with open(sf) as f:
        scores = json.load(f)
    
    rank = sf.split("rank_")[1][:3] if "rank_" in sf else "?"
    ptm = scores.get("ptm", scores.get("pTM", 0))
    plddt = scores.get("plddt", [])
    mean_plddt = sum(plddt) / len(plddt) if plddt else 0
    
    all_scores.append({"rank": rank, "pTM": ptm, "mean_pLDDT": mean_plddt, "plddt_per_residue": plddt})
    print(f"  Rank {rank}: pTM = {ptm:.3f} | mean pLDDT = {mean_plddt:.1f}")

print(f"\n--- Best model (Rank 001) ---")
best = all_scores[0]
print(f"  pTM = {best['pTM']:.3f}  ({'Good ✓' if best['pTM'] > 0.7 else 'Low — investigate'})")
print(f"  Mean pLDDT = {best['mean_pLDDT']:.1f}  ({'High confidence ✓' if best['mean_pLDDT'] > 70 else 'Low confidence'})")

## Cell 6 — Plot pLDDT per residue

This shows confidence at every position along the sequence. High = well-structured, Low = disordered.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

plddt = best["plddt_per_residue"]
residues = list(range(1, len(plddt) + 1))

fig, ax = plt.subplots(figsize=(12, 4))

# Color bars by pLDDT value
colors = []
for p in plddt:
    if p >= 90: colors.append("#003f7e")     # Very high — dark blue
    elif p >= 70: colors.append("#5fa8d3")   # High — light blue
    elif p >= 50: colors.append("#f0d43a")   # Low — yellow
    else: colors.append("#f06c00")            # Very low — orange

ax.bar(residues, plddt, color=colors, width=1.0, edgecolor='none')
ax.axhline(90, color='navy', linestyle='--', alpha=0.5, linewidth=1, label='90 (very high confidence)')
ax.axhline(70, color='steelblue', linestyle='--', alpha=0.5, linewidth=1, label='70 (high confidence)')
ax.axhline(50, color='goldenrod', linestyle='--', alpha=0.5, linewidth=1, label='50 (low confidence)')

ax.set_xlabel("Residue position", fontsize=12)
ax.set_ylabel("pLDDT", fontsize=12)
ax.set_title(f"PD-L1 pLDDT (AlphaFold2) — mean = {best['mean_pLDDT']:.1f}", fontsize=13)
ax.set_ylim(0, 100)
ax.legend(fontsize=9)

# Legend patches for colors
legend_patches = [
    mpatches.Patch(color='#003f7e', label='≥90 (very high)'),
    mpatches.Patch(color='#5fa8d3', label='70–90 (high)'),
    mpatches.Patch(color='#f0d43a', label='50–70 (low)'),
    mpatches.Patch(color='#f06c00', label='<50 (very low)')
]
ax.legend(handles=legend_patches, loc='lower right', fontsize=9)

plt.tight_layout()
plt.savefig("pdl1_plddt_plot.png", dpi=150, bbox_inches='tight')
plt.show()

# Statistics
plddt_arr = np.array(plddt)
print(f"Residues with pLDDT > 90: {(plddt_arr > 90).sum()} / {len(plddt)} ({(plddt_arr > 90).mean()*100:.0f}%)")
print(f"Residues with pLDDT > 70: {(plddt_arr > 70).sum()} / {len(plddt)} ({(plddt_arr > 70).mean()*100:.0f}%)")
print(f"Residues with pLDDT < 50: {(plddt_arr < 50).sum()} / {len(plddt)} ({(plddt_arr < 50).mean()*100:.0f}%)")

## Cell 7 — Plot the PAE matrix

The PAE (Predicted Aligned Error) matrix shows confidence in the *relative positions* of residue pairs. Dark blue = confident, yellow = uncertain.

In [ ]:
# Load PAE from the best model's score file
best_score_file = score_files[0]
with open(best_score_file) as f:
    scores = json.load(f)

pae = scores.get("pae", None)

if pae is not None:
    pae_array = np.array(pae)
    
    fig, ax = plt.subplots(figsize=(7, 6))
    im = ax.imshow(pae_array, cmap="Blues_r", vmin=0, vmax=30)
    plt.colorbar(im, ax=ax, label="Predicted aligned error (Å)")
    ax.set_title("PD-L1 PAE Matrix (AlphaFold2)", fontsize=13)
    ax.set_xlabel("Residue j", fontsize=11)
    ax.set_ylabel("Residue i", fontsize=11)
    plt.tight_layout()
    plt.savefig("pdl1_pae_matrix.png", dpi=150, bbox_inches='tight')
    plt.show()
    
    print(f"PAE matrix shape: {pae_array.shape}")
    print(f"Mean PAE (off-diagonal): {np.mean(pae_array):.1f} Å")
    print(f"For a single chain protein, the whole matrix should be dark blue.")
    print(f"High PAE values (> 10 Å) typically appear only at the diagonal extremes (termini).")
else:
    print("PAE not found in score file. This may depend on the ColabFold version.")

## Cell 8 — Visualize the predicted structure

In [ ]:
import py3Dmol
import glob

# Find the best-ranked PDB model
pred_pdb_files = sorted(glob.glob("colabfold_output/*rank_001*.pdb"))

if not pred_pdb_files:
    pred_pdb_files = sorted(glob.glob("colabfold_output/*.pdb"))

print(f"Loading: {pred_pdb_files[0]}")

with open(pred_pdb_files[0]) as f:
    pred_pdb_str = f.read()

view = py3Dmol.view(width=750, height=500)
view.addModel(pred_pdb_str, "pdb")

# Color by pLDDT (stored in B-factor column of AF2 predictions)
view.setStyle({"cartoon": {"colorscheme": {"prop": "b", "gradient": "roygb", "min": 50, "max": 90}}})

view.zoomTo()
view.setBackgroundColor("white")
view.show()

print("Color scheme: blue = high pLDDT (confident), red = low pLDDT (uncertain)")
print("For PD-L1, most of the structure should be blue.")

## Cell 9 — Compare AF2 prediction to X-ray structure

Let's overlay the AF2 prediction with the experimental X-ray structure (5C3T) to see how well they agree.

In [ ]:
import requests

# Download 5C3T experimental structure
xray_pdb = requests.get("https://files.rcsb.org/download/5C3T.pdb").text

view_compare = py3Dmol.view(width=750, height=500)

# Load AF2 prediction
view_compare.addModel(pred_pdb_str, "pdb")
view_compare.setStyle({"model": 0}, {"cartoon": {"color": "#B76E79"}})

# Load X-ray structure (chain A only)
view_compare.addModel(xray_pdb, "pdb")
view_compare.setStyle({"model": 1, "chain": "A"}, {"cartoon": {"color": "#3c5b6f"}})

view_compare.zoomTo()
view_compare.setBackgroundColor("white")
view_compare.show()

print("Rose = AlphaFold2 prediction | Blue = X-ray crystal structure (5C3T)")
print("If they overlap well, AlphaFold2 successfully predicted PD-L1's structure.")
print("Note: They may not be perfectly aligned without structural superposition.")

## Cell 10 — Download your results

In [ ]:
from google.colab import files
import os
import glob

# Download the best PDB model
for f in pred_pdb_files[:1]:
    files.download(f)

# Download the pLDDT plot
if os.path.exists("pdl1_plddt_plot.png"):
    files.download("pdl1_plddt_plot.png")

print("Files downloaded to your computer.")

## Cell 11 — Questions to answer

Fill in your answers below.

**Q1:** What is the pTM score for the best model? Is it above 0.7 (the good threshold)?

_Your answer:_

**Q2:** What is the mean pLDDT? What fraction of residues have pLDDT > 90?

_Your answer:_

**Q3:** In the pLDDT plot, are there any dips (low confidence regions)? Where are they — beginning, middle, or end of the sequence?

_Your answer:_

**Q4:** Looking at the PAE matrix — is it mostly dark blue (low PAE)? What does that tell you?

_Your answer:_

**Q5:** In the visual comparison (Cell 9), do the AF2 prediction and X-ray structure appear to have the same overall fold? How well do they match?

_Your answer:_

---
## Summary

In this notebook you:
- Installed and ran ColabFold on the PD-L1 sequence
- Interpreted pLDDT and pTM confidence scores
- Visualized the per-residue pLDDT profile
- Plotted the PAE matrix
- Compared the AF2 prediction to the experimental X-ray structure

**Next:** Notebook 03 — MSA comparison (with vs. without evolutionary information)